# Leeds parking fine-tuning — complete Colab run

Run every cell from top to bottom. This notebook mounts Drive, clones the exact GitHub repository, downloads only the 100 required Git LFS TIFFs, restores or builds the masks, downloads the released checkpoint, fine-tunes, evaluates, and saves durable outputs to Drive.

The only possible manual action is selecting `leeds_grid.gpkg` and `leeds_manual.gpkg` if they cannot be found in Drive and no prepared-data cache exists.

## 1. Configuration, Drive and GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/hou1020/Parking.git'
BRANCH = 'main'
REPO = Path('/content/Parking')
DRIVE_RUN = Path('/content/drive/MyDrive/Parking_finetuning_run')
DATA_CACHE = DRIVE_RUN / 'prepared_data'
DRIVE_RUN.mkdir(parents=True, exist_ok=True)

EPOCHS = 6
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 4
NUM_WORKERS = 2
FORCE_RETRAIN = False  # set True only when deliberately replacing an existing run

import torch
assert torch.cuda.is_available(), (
    'No CUDA GPU. In Colab choose Runtime > Change runtime type > GPU, then rerun.'
)
print('GPU:', torch.cuda.get_device_name(0))
print('Persistent output:', DRIVE_RUN)

## 2. Install runtime dependencies

In [ ]:
if shutil.which('git-lfs') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'git-lfs'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'huggingface_hub', 'pandas', 'scipy',
                'Pillow', 'geopandas', 'tifffile', 'matplotlib'], check=True)
print('Dependencies installed.')

## 3. Clone/update the repository and fetch the 100 source TIFFs

LFS smudging is disabled during clone so Colab does not download unrelated large files. The following cell pulls only `parking-lot-mapping-tool/files/tif/**`.

In [ ]:
env = os.environ.copy()
env['GIT_LFS_SKIP_SMUDGE'] = '1'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    REPO_URL, str(REPO)], env=env, check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)

subprocess.run(['git', '-C', str(REPO), 'lfs', 'install', '--local'], check=True)
subprocess.run(['git', '-C', str(REPO), 'lfs', 'pull',
                '--include=parking-lot-mapping-tool/files/tif/**'], check=True)

FT = REPO / 'fine-tuning'
TIF_ROOT = REPO / 'parking-lot-mapping-tool/files/tif'
tifs = sorted(TIF_ROOT.rglob('*.tif'))
pointer_like = [p for p in tifs if p.stat().st_size < 1_000_000]
assert len(tifs) == 100, f'Expected 100 TIFFs, found {len(tifs)}'
assert not pointer_like, f'LFS TIFFs were not downloaded: {pointer_like[:3]}'
print('Repository:', REPO)
print('Commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('Real source TIFFs:', len(tifs))

## 4. Restore or build patch masks

Generated masks and `patch_index.csv` are intentionally not stored in Git. On the first run this cell searches common Drive locations for the two manual GPKGs. If they are absent, Colab opens one upload dialog; select both files together. The resulting 26 MB prepared dataset is cached in Drive, so later runtimes restore it automatically.

In [ ]:
def prepared_ok(root):
    root = Path(root)
    return (root / 'patch_index.csv').exists() and \
           len(list((root / 'patches/train/Masks').glob('*.png'))) == 2256 and \
           len(list((root / 'patches/test/Masks').glob('*.png'))) == 3200

if prepared_ok(FT):
    print('Prepared data already present in the cloned repository runtime.')
elif prepared_ok(DATA_CACHE):
    shutil.copy2(DATA_CACHE / 'patch_index.csv', FT / 'patch_index.csv')
    shutil.copytree(DATA_CACHE / 'patches', FT / 'patches', dirs_exist_ok=True)
    if (DATA_CACHE / 'alignment_check.png').exists():
        shutil.copy2(DATA_CACHE / 'alignment_check.png', FT / 'alignment_check.png')
    print('Restored prepared masks from Drive cache.')
else:
    manual_dir = Path('/content/manual')
    manual_dir.mkdir(parents=True, exist_ok=True)
    names = ('leeds_grid.gpkg', 'leeds_manual.gpkg')
    search_roots = [
        Path('/content/drive/MyDrive/manual'),
        Path('/content/drive/MyDrive/dissertation/manual'),
        Path('/content/drive/MyDrive/Parking/manual'),
    ]
    found = {}
    for base in search_roots:
        for name in names:
            candidate = base / name
            if candidate.exists():
                found[name] = candidate
    if len(found) < 2:
        from google.colab import files
        print('Select leeds_grid.gpkg and leeds_manual.gpkg together.')
        uploaded = files.upload()
        for name in names:
            if name in uploaded:
                source = Path('/content') / name
                source.write_bytes(uploaded[name])
                found[name] = source
    missing = [name for name in names if name not in found]
    assert not missing, f'Missing manual inputs: {missing}'
    for name, source in found.items():
        shutil.copy2(source, manual_dir / name)

    subprocess.run([sys.executable, str(FT / 'make_split.py')], cwd=FT, check=True)
    subprocess.run([sys.executable, str(FT / 'make_patches.py')], cwd=FT, check=True)
    subprocess.run([sys.executable, str(FT / 'check_alignment.py')], cwd=FT, check=True)
    assert prepared_ok(FT), 'Prepared data counts failed after generation'

    DATA_CACHE.mkdir(parents=True, exist_ok=True)
    shutil.copy2(FT / 'patch_index.csv', DATA_CACHE / 'patch_index.csv')
    shutil.copytree(FT / 'patches', DATA_CACHE / 'patches', dirs_exist_ok=True)
    shutil.copy2(FT / 'alignment_check.png', DATA_CACHE / 'alignment_check.png')
    print('Built data and cached it in Drive:', DATA_CACHE)

assert prepared_ok(FT)
print('Train masks:', len(list((FT / 'patches/train/Masks').glob('*.png'))))
print('Test masks:', len(list((FT / 'patches/test/Masks').glob('*.png'))))

In [ ]:
from IPython.display import display, Image as DisplayImage
if (FT / 'alignment_check.png').exists():
    display(DisplayImage(filename=str(FT / 'alignment_check.png'), width=1100))
else:
    print('Alignment image was not cached; numeric/data checks still passed.')

## 5. Download and validate the released checkpoint

The real 1.017 GB checkpoint is saved in Drive, not in the ephemeral runtime and not at the repository's 135-byte Git LFS pointer.

In [ ]:
from huggingface_hub import hf_hub_download

ZERO_SHOT = DRIVE_RUN / 'best_model.ckpt'
if not ZERO_SHOT.exists() or ZERO_SHOT.stat().st_size < 1_000_000_000:
    hf_hub_download(
        repo_id='UTEL-UIUC/SegFormer-large-parking',
        filename='best_model.ckpt',
        local_dir=str(DRIVE_RUN),
    )
assert ZERO_SHOT.stat().st_size > 1_000_000_000, 'Checkpoint download is incomplete'
print(f'Checkpoint: {ZERO_SHOT} ({ZERO_SHOT.stat().st_size / 1e9:.3f} GB)')

## 6. Fine-tune

The best genuinely fine-tuned epoch is written directly to Drive. If Colab reports CUDA out of memory, change `TRAIN_BATCH_SIZE` in the first cell from 2 to 1, reconnect the runtime, and run all cells again. Existing completed training is skipped unless `FORCE_RETRAIN=True`.

In [ ]:
FINETUNED = DRIVE_RUN / 'finetuned.ckpt'
if FORCE_RETRAIN or not FINETUNED.exists() or FINETUNED.stat().st_size < 250_000_000:
    command = [
        sys.executable, str(FT / 'finetune.py'),
        '--patches', str(FT / 'patches'),
        '--index', str(FT / 'patch_index.csv'),
        '--tif-root', str(TIF_ROOT),
        '--ckpt', str(ZERO_SHOT),
        '--out', str(FINETUNED),
        '--epochs', str(EPOCHS),
        '--batch-size', str(TRAIN_BATCH_SIZE),
        '--num-workers', str(NUM_WORKERS),
    ]
    subprocess.run(command, cwd=FT, check=True)
else:
    print('Using existing fine-tuned checkpoint:', FINETUNED)

assert FINETUNED.exists() and FINETUNED.stat().st_size > 250_000_000
for name in ('fit_val_split.csv', 'finetune_log.csv'):
    source = FT / name
    if source.exists():
        shutil.copy2(source, DRIVE_RUN / name)
print(f'Fine-tuned checkpoint: {FINETUNED.stat().st_size / 1e9:.3f} GB')

## 7. Evaluate zero-shot and fine-tuned models on the same held-out cells

In [ ]:
EVALUATION = DRIVE_RUN / 'evaluation.csv'
BOUNDARIES = DRIVE_RUN / 'boundary_bands.csv'
command = [
    sys.executable, str(FT / 'evaluate.py'),
    '--patches', str(FT / 'patches'),
    '--index', str(FT / 'patch_index.csv'),
    '--tif-root', str(TIF_ROOT),
    '--zero-shot', str(ZERO_SHOT),
    '--finetuned', str(FINETUNED),
    '--batch-size', str(EVAL_BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--out', str(EVALUATION),
    '--boundary-out', str(BOUNDARIES),
]
subprocess.run(command, cwd=FT, check=True)
assert EVALUATION.exists() and BOUNDARIES.exists()
print('Evaluation outputs saved in:', DRIVE_RUN)

## 8. Inspect the final outputs

In [ ]:
import pandas as pd

if (DRIVE_RUN / 'finetune_log.csv').exists():
    print('Training history')
    display(pd.read_csv(DRIVE_RUN / 'finetune_log.csv'))
print('Held-out comparison')
display(pd.read_csv(EVALUATION))
print('Boundary sensitivity')
display(pd.read_csv(BOUNDARIES))

print('Durable files:')
for path in sorted(DRIVE_RUN.iterdir()):
    if path.is_file():
        print(f'  {path.name:<24} {path.stat().st_size / 1e6:,.1f} MB')